# Predicting Cardiovascular Disease from Routine Health Check-up Data
### DS3000 — Introduction to Machine Learning · Course Project · Fall 2026

**Group 1:** Kristina, Romy, Rakan, Brandon
**Dataset:** #19 — Cardiovascular Disease (OpenML, dataset ID 45547)

---

### Problem statement
Cardiovascular disease is one of the leading causes of death worldwide, and many cases can be prevented
when people at risk are identified early. In this project, we investigate whether the presence of
cardiovascular disease can be predicted from information that is routinely collected during a standard
medical check-up, such as age, blood pressure, cholesterol and lifestyle habits.

### Task
This is a **supervised binary classification** problem. Each row in the dataset represents one patient,
and the target variable `cardio` records whether that patient has cardiovascular disease (`1`) or not (`0`).
The models we build will learn from patients whose outcome is known, and will then be evaluated on their
ability to correctly classify patients they have not seen before.

### Notebook structure
1. Setup
2. Data loading and description
3. Exploratory data analysis *(to be added)*
4. Data cleaning *(to be added)*
5. Train / validation / test split *(to be added)*
6. Baseline models *(to be added)*

## 1. Setup

All libraries used in this notebook are imported in a single cell so that every dependency is visible in one
place and the notebook can be run from top to bottom without errors.

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations on arrays |
| `pandas` | Loading and manipulating tabular data (DataFrames) |
| `matplotlib` / `seaborn` | Data visualisation |

We also define a single random seed, `SEED = 42`. Several steps later in the project involve randomness,
such as shuffling the data before splitting it or initialising certain models. Fixing the seed ensures
that these steps produce identical results every time the notebook is run, which makes our results
**reproducible** for every group member and for anyone evaluating the project. The specific value has no
special meaning; what matters is that the same value is used consistently throughout.

In [1]:
# Core libraries
import numpy as np                # numerical operations
import pandas as pd               # tabular data (DataFrames)
import matplotlib.pyplot as plt   # plotting
import seaborn as sns             # statistical plots built on matplotlib

# Random seed used for every random operation in this notebook (for reproducibility)
SEED = 42

print("Libraries imported successfully. Random seed =", SEED)

Libraries imported successfully. Random seed = 42


## 2. Data Loading and Description

### 2.1 Loading the dataset
The dataset is stored in the repository as `data/cardio.arff`. It was obtained from OpenML
(https://www.openml.org/d/45547) and is kept locally so that the notebook can run without an internet connection.

The file is in **ARFF format** (Attribute-Relation File Format), a plain-text format used by OpenML. It begins
with a 47-line header that describes the dataset and its attributes, and the patient records begin on line 48.
For this reason, the header is skipped when reading the file, and the column names are supplied manually
in the same order in which they are declared in the header.

A **relative path** (`data/cardio.arff`) is used so that the notebook runs on any machine where the repository
has been cloned, without needing to change any file locations. When the notebook is run in an environment
that does not contain the repository files, such as Google Colab, the same file is read directly from the
project's GitHub repository instead. In both cases the data is identical, so the results are the same.

In [2]:
import os

# Column names, in the order they are declared in the ARFF header
column_names = ["id", "age", "gender", "height", "weight", "ap_hi", "ap_lo",
                "cholesterol", "gluc", "smoke", "alco", "active", "cardio"]

# Use the local copy in the repository if it exists (e.g. running in Jupyter on our own laptops).
# Otherwise (e.g. running in Google Colab, which has no access to our local files),
# read the same file directly from our GitHub repository.
DATA_PATH = "data/cardio.arff"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "https://raw.githubusercontent.com/Kristina-Kojic/3000-AISE/main/data/cardio.arff"

# Skip the 47-line ARFF header; the file has no header row, so column names are supplied manually
df = pd.read_csv(DATA_PATH, skiprows=47, header=None, names=column_names)

print("Data loaded from:", DATA_PATH)
print("Number of rows (patients):", df.shape[0])
print("Number of columns:        ", df.shape[1])
df.head()

Data loaded from: https://raw.githubusercontent.com/Kristina-Kojic/3000-AISE/main/data/cardio.arff
Number of rows (patients): 70000
Number of columns:         13


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110.0,80.0,1,1,0,0,1,0
1,1,20228,1,156,85.0,140.0,90.0,3,1,0,0,1,1
2,2,18857,1,165,64.0,130.0,70.0,3,1,0,0,0,1
3,3,17623,2,169,82.0,150.0,100.0,1,1,0,0,1,1
4,4,17474,1,156,56.0,100.0,60.0,1,1,0,0,0,0


### 2.2 Data types
Before any analysis, we confirm that every column was read as a numeric type. If a column had been read as
text, this would indicate a loading problem (for example, a stray header line being treated as data).

In [3]:
df.dtypes

id               int64
age              int64
gender           int64
height           int64
weight         float64
ap_hi          float64
ap_lo          float64
cholesterol      int64
gluc             int64
smoke            int64
alco             int64
active           int64
cardio           int64
dtype: object

### 2.3 Class balance of the target variable
We count the number of patients in each class of `cardio`. The class balance determines which evaluation
metrics are meaningful: when one class is much more common than the other, accuracy alone becomes misleading,
because a model could achieve a high accuracy simply by always predicting the majority class.

In [4]:
class_counts = df["cardio"].value_counts()
class_percent = df["cardio"].value_counts(normalize=True) * 100

pd.DataFrame({"Count": class_counts, "Percentage (%)": class_percent.round(2)})

,Count,Percentage (%)
cardio,,
0,35021,50.03
1,34979,49.97


### 2.4 Description of the variables
The dataset description groups the features into three types: **objective** (factual information),
**examination** (results of the medical examination) and **subjective** (information reported by the patient).

| Column | Description | Feature type | Unit / values |
|---|---|---|---|
| `id` | Record identifier | — | Integer. Not a predictive feature; it will be removed during cleaning |
| `age` | Age of the patient | Objective | **Days** (to be converted to years during cleaning) |
| `gender` | Gender of the patient | Objective | Categorical code: 1 or 2 |
| `height` | Height | Objective | cm |
| `weight` | Weight | Objective | kg |
| `ap_hi` | Systolic blood pressure (upper value) | Examination | mmHg |
| `ap_lo` | Diastolic blood pressure (lower value) | Examination | mmHg |
| `cholesterol` | Cholesterol level | Examination | 1 = normal, 2 = above normal, 3 = well above normal |
| `gluc` | Glucose level | Examination | 1 = normal, 2 = above normal, 3 = well above normal |
| `smoke` | Smoking status | Subjective | 0 = no, 1 = yes |
| `alco` | Alcohol intake | Subjective | 0 = no, 1 = yes |
| `active` | Physical activity | Subjective | 0 = no, 1 = yes |
| `cardio` | **Target:** presence of cardiovascular disease | — | 0 = absent, 1 = present |

### 2.5 Initial observations
- The dataset contains **70,000 patients** and **13 columns**: 11 input features, one identifier (`id`)
  and one target variable (`cardio`).
- All columns were loaded as numeric types, confirming that the file was read correctly.
- The target is **almost perfectly balanced** (approximately 50% of patients with the disease and 50% without).
  As a result, accuracy is a fair summary metric for this problem. However, because failing to detect a patient
  who has the disease (a false negative) is more serious in a medical screening context than a false alarm,
  recall will also be an important metric when comparing models.
- `cholesterol` and `gluc` are **ordinal** variables: their values follow a natural order
  (normal < above normal < well above normal), so their numeric codes can be used directly.
- Three features (`smoke`, `alco`, `active`) are **self-reported** by the patient, which means they may be less
  reliable than measured values. This is a potential limitation that will be revisited later in the project.